# Validate Reusable Ingestion Framework

This notebook validates the AeroPulse reusable ingestion framework before it is used by entity-specific Bronze pipelines.

The validation covers:

- Source delivery discovery
- Ingestion registry status checks
- Source format reading
- Bronze metadata generation
- Bronze Delta writing

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/ingestion/delivery_discovery.py

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/ingestion/ingestion_registry.py

In [0]:
%run /Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/ingestion/bronze_ingestion.py

In [0]:
ENVIRONMENT = "dev"

CATALOG = "workspace"

SCHEMA = f"aeropulse_{ENVIRONMENT}"

SOURCE_SYSTEM = "erp"

SOURCE_ENTITY = "airlines"

FILE_FORMAT = "csv"

SOURCE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing/"
    f"{SOURCE_SYSTEM}/{SOURCE_ENTITY}"
)

BRONZE_TABLE = (
    f"{CATALOG}.{SCHEMA}.bronze_{SOURCE_ENTITY}"
)

INGESTION_FILE_REGISTRY_TABLE = (
    f"{CATALOG}.{SCHEMA}.ingestion_file_registry"
)

print(f"Source path: {SOURCE_PATH}")
print(f"Bronze table: {BRONZE_TABLE}")
print(f"Registry: {INGESTION_FILE_REGISTRY_TABLE}")

In [0]:
delivery_paths = discover_source_deliveries(
    dbutils=dbutils,
    source_path=SOURCE_PATH,
    source_entity=SOURCE_ENTITY,
)

print("Discovered deliveries:")

for delivery_path in delivery_paths:
    print(delivery_path)

In [0]:
for delivery_path in delivery_paths:

    processed = is_delivery_processed(
        spark=spark,
        registry_table=INGESTION_FILE_REGISTRY_TABLE,
        source_delivery_path=delivery_path,
    )

    print(
        f"{delivery_path} "
        f"→ Already processed: {processed}"
    )

In [0]:
validation_delivery_path = delivery_paths[0]

In [0]:
validation_source_df = read_source_delivery(
    spark=spark,
    source_delivery_path=validation_delivery_path,
    file_format=FILE_FORMAT,
)

print(
    f"Records read: "
    f"{validation_source_df.count()}"
)

In [0]:
VALIDATION_PIPELINE_RUN_ID = "framework-validation"

In [0]:
validation_bronze_df = (
    validation_source_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file_path",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "_pipeline_run_id",
        F.lit(VALIDATION_PIPELINE_RUN_ID)
    )
)

display(validation_bronze_df)